# Get Donegal Townland Population Data
This notebook prepares population data for Donegal (2016 census) to be applied to the road network graph as edge weights. 

Initally data for Donegal townlands is extracted from overall data (soure: https://www.cso.ie/en/census/census2016reports/census2016smallareapopulationstatistics/). Then the OSM API is queried to get approximate latitude and longitude co-ordinates for each townland.

A helper module - population_helper.py - contains functions used throughout this task.

Extract Donegal data from source xls file into a pandas dataframe, combine townland and town columns, remove unnecessary information and write data to a csv file:

In [ ]:
import pandas as pd

import config
import graph.helpers.population_helper as helper

In [ ]:
donegal_data = helper.extract_county_townlands_from_source_data()
donegal_data.head()

2020 townland records left from soure data:

In [ ]:
len(donegal_data)

Now using the query function from OSMPythonTools.nominatim an attempt to get OSM coordinates for each townland row of the dataframe is made. Any successful matches are added as new columns "lat" and "lng" to the dataframe:

In [ ]:
donegal_data = donegal_data.apply(
    helper.lookup_osm_coordinates, args=("townland",), axis=1
)

Save this to a new csv:

In [ ]:
donegal_data.to_csv(
    f"{config.population_data_path}/donegal_townlands_with_coordinates.csv"
)

Check which rows didn't get lat/long. values and save to csv:

In [ ]:
donegal_data = pd.read_csv(
    f"{config.population_data_path}/donegal_townlands_with_coordinates.csv",
    na_values=None,
)
len(donegal_data)

In [ ]:
no_coords = donegal_data[donegal_data["lat"].isnull()]
len(no_coords)

In [ ]:
no_coords = no_coords.mask(no_coords.isna(), other=None)
no_coords.to_csv(f"{config.population_data_path}/donegal_townlands_no_coordinates.csv")
no_coords

Query API with combined townland and town values to see if that finds any more:

In [ ]:
no_coords["address"] = no_coords[["townland", "town"]].agg(", ".join, axis=1)

checked_coords = no_coords.apply(
    helper.lookup_osm_coordinates, args=("address",), axis=1
)

Again check which rows have no co-ordinates

In [ ]:
no_coords = checked_coords[checked_coords["lat"].isnull()]
len(no_coords)

Some reduction: from 135 to 96. These last ones need to be checked manually. Adding newly found co-ordinates to overall dataset and saving remaining missing values to disk.

In [ ]:
donegal_data.update(checked_coords)
donegal_data = donegal_data[["townland", "town", "population", "lat", "lng"]]
donegal_data

In [ ]:
len(donegal_data[donegal_data["lat"].isnull()])

In [ ]:
no_coords.to_csv(f"{config.population_data_path}/donegal_townlands_manual.csv")

Missing co-ordinates found by manually checking OSM and Google Maps for latitude and longitude values. Loading the updated csv file, check that co-ordinates are all there:

In [ ]:
updated_missings = pd.read_csv(
    f"{config.population_data_path}/donegal_townlands_manual_done.csv", na_values=None
)
updated_missings = updated_missings[["townland", "town", "population", "lat", "lng"]]

updated_missings

Update main dataset with updated co-ords:

In [ ]:
updated_donegal_data = donegal_data.merge(
    updated_missings, on=["townland", "town", "population"], how="left"
)
updated_donegal_data["lat_x"].fillna(updated_donegal_data["lat_y"], inplace=True)
updated_donegal_data["lng_x"].fillna(updated_donegal_data["lng_y"], inplace=True)
updated_donegal_data = updated_donegal_data[
    ["townland", "town", "population", "lat_x", "lng_x"]
]
updated_donegal_data = updated_donegal_data.rename(
    columns={"lat_x": "lat", "lng_x": "lng"}
)

updated_donegal_data

In [ ]:
len(updated_donegal_data[updated_donegal_data["lat"].isnull()])

In [ ]:
len(updated_donegal_data[updated_donegal_data["lng"].isnull()])

Save csv with all co-ordinates:

In [ ]:
updated_donegal_data.to_csv(config.population_csv)